WARNING: Notebook was written during development. May not work as intended. Retained here only for completion and future reference.

In [ ]:
import jax
import jax.numpy as jnp
import matplotlib
import matplotlib.pyplot as plt
import numpy as np

import crops.src.bc as _bc
import crops.src.fe_cosserat as _fe
import crops.src.mesher as _mesher
import crops.src.solver as _solver
import crops.src.utils as _utils
import crops.src.viz as _viz
import crops.tools.geom as _geom

plt.rcParams.update(_viz.high_res_plot_settings)
jax.config.update("jax_enable_x64", True)

### Geometry of rod

In [ ]:
rod_length = 6.0e-2  # 60 mm stent length

tube_radius = 5.0e-3  # 10 mm diameter stent
fiber_radius = 2.0e-4  # 0.4 mm diameter wire

num_fibers = 144  # div along the circumference
num_elems = 288  # div along the length
num_nodes = num_elems + 1


num_cells_circum, num_cells_axial = 4, 8  # unit cells of stent along circum and len

### Stent mask

In [ ]:
fiber_offsets = _geom.annular_fiber_offsets(
  num_elems=num_elems,
  num_fibers=num_fibers,
  semi_axis_d1=tube_radius,
)

mask = _geom.create_stent_mask(
  unit_cell=plt.imread("./crops/data/stent_designs/to.png"),
  num_elems=num_elems,
  num_fibers=num_fibers,
  num_cells_axial=12,
  num_cells_circum=6,
)

area_per_fiber = np.pi * fiber_radius**2
fiber_areas = mask * area_per_fiber


In [ ]:
sector_span = 2.0 * np.pi / num_fibers
wall_thickness = area_per_fiber / (sector_span * tube_radius)

fiber_angles = np.arange(num_fibers) * sector_span
fiber_start_angles = np.tile(fiber_angles - 0.5 * sector_span, (num_elems, 1))
fiber_end_angles = np.tile(fiber_angles + 0.5 * sector_span, (num_elems, 1))
fiber_inner_radii = np.full((num_elems, num_fibers), tube_radius - 0.5 * wall_thickness)
fiber_outer_radii = np.full((num_elems, num_fibers), tube_radius + 0.5 * wall_thickness)


In [ ]:
rod = _mesher.build_rod_geometry(
  analytical_curve=lambda s: jnp.array([s, 0.0, 0.0]),
  parametric_coords=jnp.linspace(0.0, rod_length, num_nodes),
  fiber_offsets=jnp.asarray(fiber_offsets),
  fiber_areas=jnp.asarray(fiber_areas),
  fiber_mass_density=jnp.asarray(mask * 1100.0),
  fiber_centroid_moi=jnp.zeros((num_elems, num_fibers, 2)),  # wires as points
  seed_binormal=jnp.array([0.0, 0.0, 1.0]),
)

In [ ]:
fig, ax = plt.subplots(figsize=(9, 3))
ax.imshow(mask.T, aspect="auto", origin="lower", cmap="gray")
ax.set_xlabel("element index (axial)")
ax.set_ylabel("fiber index (circumferential)")
ax.set_title("unrolled stent surface")
plt.show()

## Material

In [ ]:
shear_modulus = 1e6  # 1 MPa

### BC

In [ ]:
num_steps = 30
clamped_node = np.array([0])
tip_node = np.array([num_nodes - 1])
tip_load = 0.10  # N

clamped_dofs = (
  _utils.Translation.X,
  _utils.Translation.Y,
  _utils.Translation.Z,
  _utils.Rotation.THETA_X,
  _utils.Rotation.THETA_Y,
  _utils.Rotation.THETA_Z,
)
clamp = [
  _bc.DirichletBC(
    values=(dof, [np.array([0.0]) for _ in range(num_steps)]),
    node_targets=[clamped_node for _ in range(num_steps)],
    name=f"clamp_{dof}",
  )
  for dof in clamped_dofs
]

loads = tip_load * (np.arange(1, num_steps + 1) / num_steps) ** 2
load = _bc.NeumannBC(
  values=(_utils.Translation.Z, [np.array([p]) for p in loads]),
  node_targets=[tip_node for _ in range(num_steps)],
  name="tip_load",
)

boundary_conditions = _bc.process_boundary_conditions(rod, clamp + [load])

### Solver

In [ ]:
nr_solver = _solver.NewtonRaphson(
  solver_type=_solver.NonlinearSolvers.ARMIJO_BACKTRACKING_NEWTON_RAPHSON,
  nonlinear_settings={
    "max_iter": 300,
    "threshold": 1e-6,
    "lam_min": 0.01,
    "line_search_max_iter": 15,
    "line_search_shrink": 0.5,
    "line_search_alpha_min": 1e-6,
    "line_search_armijo_c": 1e-4,
    "abs_threshold": 1e-6,
  },
  linear_settings={"solver": _solver.LinearSolvers.SCIPY_SPARSE, "rtol": 1e-8},
)


problem = _fe.CosseratProblem(rod=rod, bc=boundary_conditions, nr_solver=nr_solver)

In [ ]:
shear_moduli = jnp.full((num_elems, num_fibers), shear_modulus)
u_star, _ = problem.solve(u0=jnp.zeros(rod.num_dofs), shear_moduli=shear_moduli)

u_nodal = np.asarray(u_star).reshape(num_nodes, rod.num_dofs_per_node)
deflection = u_nodal[:, _utils.Translation.Z.value]
print(f"tip defl. = {deflection[-1]:.6e} m   ({deflection[-1] / rod_length:.3%} of L)")

In [ ]:
is_solid = mask > 0.5

annular_kwargs = dict(
  fiber_inner_radii=fiber_inner_radii,
  fiber_outer_radii=fiber_outer_radii,
  fiber_start_angles=fiber_start_angles,
  fiber_end_angles=fiber_end_angles,
  fiber_mask=is_solid,
  num_arc_segments=2,
  edge_width=0.0,
)

ax = _viz.plot_rod_annular_sections(
  node_coords=np.asarray(rod.node_coords),
  directors=np.asarray(rod.directors_0),
  fiber_field=np.zeros_like(mask),
  fiber_field_val_range=(0.0, 1.0),
  cmap="coolwarm",
  alpha=0.25,
  **annular_kwargs,
)

In [ ]:
import matplotlib
%matplotlib qt5

coords_def, directors_def, _ = _mesher.extract_coords_and_director_from_deformation(
  rod, u_star
)

ax = _viz.plot_rod_annular_sections(
  node_coords=np.asarray(coords_def),
  directors=np.asarray(directors_def),
  fiber_field=np.ones_like(mask),
  fiber_field_val_range=(0.0, 1.0),
  cmap="coolwarm",
  alpha=1.0,
  # ax=ax,
  **annular_kwargs,
)
plt.show()